# Notebook 4: Knowledge curation from noisy observations

This notebook demonstrates the repo’s normalization layer: extract claims from messy operational notes, canonicalize entities, remove duplicates, score the resulting memories, and prepare a cleaner memory set.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.claims.extractor import ClaimExtractor
from src.normalization.duplicate_detector import DuplicateDetector
from src.normalization.entity_linker import EntityLinker
from src.normalization.mapper import ClaimToMemoryMapper
from src.normalization.scorer import ConfidenceScorer, SalienceScorer

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-04"
shutil.rmtree(memory_root, ignore_errors=True)
extractor = ClaimExtractor()
linker = EntityLinker()
detector = DuplicateDetector(threshold=0.7)
mapper = ClaimToMemoryMapper()
salience = SalienceScorer()
confidence = ConfidenceScorer()

In [ ]:
raw_notes = [
    "Acme Payments failed closed after the certificate rotation. Customers saw checkout errors in London.",
    "ACME Payments failed closed after certificate rotation and checkout failed for enterprise customers in London.",
    "Later, Acme payments restored service after the previous certificate bundle was reapplied.",
]

claims = []
for index, note in enumerate(raw_notes, start=1):
    extracted = extractor.extract(note, source_ref=f"note-{index}")
    for claim in extracted:
        claim.entities = linker.extract_and_link(claim.claim_text)
    claims.extend(extracted)

show({
    "raw_claim_count": len(claims),
    "claims": [claim.model_dump(mode="json") for claim in claims],
})

In [ ]:
duplicate_pairs = [
    {
        "left": left.claim_text,
        "right": right.claim_text,
    }
    for left, right in detector.find_duplicates(claims)
]
deduplicated_claims = detector.deduplicate(claims)
memories = mapper.map_many(deduplicated_claims, session_id="curation-demo")
ranked_memories = [
    {
        **memory.model_dump(mode="json"),
        "salience_score": salience.score(memory),
        "confidence_score": confidence.score(memory),
    }
    for memory in memories
]
ranked_memories.sort(key=lambda item: item["salience_score"], reverse=True)
show({
    "duplicate_pairs": duplicate_pairs,
    "deduplicated_claim_count": len(deduplicated_claims),
    "ranked_memories": ranked_memories,
})

## What this notebook demonstrates

- turning noisy notes into structured claims
- canonicalizing entity mentions with the linker
- removing duplicate/near-duplicate claims
- mapping curated claims into memory candidates
- scoring memory candidates for downstream prioritization